# Mesh 10 · A cloth that falls: simulation on a mesh

**Application, advanced.** Companion to [Week 05 §12](../README.md#12-mesh-in-architecture-four-applications)
and `mesh_applications/cloth_simulation.py`. Uses notebooks 01–04, and
pairs with [`07_form_finding`](07_form_finding.ipynb).

![A cloth falling onto a sphere at three moments in time](../figures/app-cloth.svg)

Form finding solved for a shape in one step. Here the shape is found the
slow way, the way the material does it: **let it move until it stops**.
Vertices are particles with weight. Edges are threads that refuse to
change length. That is the whole model, and it drapes fabric over a
sphere, hangs a curtain, or sags a membrane between masts.

| Step | What is added |
| --- | --- |
| 1 | one particle, falling |
| 2 | one thread between two particles |
| 3 | a whole mesh of threads |
| 4 | something to land on |
| 5 | the honesty check: did it stretch? |
| 6 | pinned corners: a hanging canopy |

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in its lecture folder or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

import numpy as np
from compas.datastructures import Mesh

## 1 · One particle, falling

Time moves in small steps of `dt`. In each step gravity changes the
velocity, and the velocity changes the position. Nothing else.

In [ ]:
dt, gravity = 1 / 60, -9.81
z, velocity = 2.0, 0.0
heights = [z]
for step in range(36):
    velocity += gravity * dt
    z += velocity * dt
    heights.append(z)
print(f"after {36 * dt:.1f} s: z = {z:.2f} m   (the formula 2 - g t^2 / 2 gives {2 + 0.5 * gravity * (36 * dt) ** 2:.2f})")
chart({"height (m)": heights}, "one particle in free fall", xlabel="steps of 1/60 s")

Close to the exact answer, not equal: stepping through time is an
approximation, and smaller steps make it better. Keep that in mind for
everything that follows.

## 2 · One thread

Two particles joined by a thread of length 1. Move one of them, and the
thread is the wrong length. **Fix it directly**: slide both particles
along the line between them, half the error each, until the length is
right again.

In [ ]:
def fix_length(a, b, rest):
    """Move two points so that they are `rest` apart, sharing the correction equally."""
    d = b - a
    length = np.linalg.norm(d)
    correction = 0.5 * (length - rest) / length * d
    return a + correction, b - correction


a, b = np.array([0.0, 0.0, 0.0]), np.array([1.6, 0.0, 0.0])
print(f"before: {np.linalg.norm(b - a):.2f} apart")
a, b = fix_length(a, b, rest=1.0)
print(f"after:  {np.linalg.norm(b - a):.2f} apart, at x = {a[0]:.2f} and x = {b[0]:.2f}")

No springs, no stiffness, no force. The thread is a **rule about
positions**, enforced by moving positions. This is *position-based
dynamics*, and it is why this simulation stays stable where a spring model
would explode.

## 3 · A mesh of threads

A square of cloth: every vertex a particle, every edge a thread at its
starting length. The mesh is used **once**, to list the edges; after that
the positions live in one array, for speed.

In [ ]:
def cloth(size=4.0, n=16, height=1.6):
    mesh = Mesh.from_meshgrid(dx=size, nx=n)
    for k in mesh.vertices():
        x, y, _ = mesh.vertex_coordinates(k)
        mesh.vertex_attributes(k, "xyz", [x - size / 2, y - size / 2, height])
    return mesh


start = cloth()
keys = list(start.vertices())
index = {k: i for i, k in enumerate(keys)}
X0 = np.array([start.vertex_coordinates(k) for k in keys])
E = np.array([(index[u], index[v]) for u, v in start.edges()])
rest = np.linalg.norm(X0[E[:, 0]] - X0[E[:, 1]], axis=1)
print(f"{len(X0)} particles, {len(E)} threads, each {rest[0]:.2f} m long")

One problem: a vertex has up to four threads, and fixing one breaks the
others. So fix them all, a little, and **repeat**. Each pass, every thread
says where it wants its two ends; every particle moves to the average of
what its threads asked for.

In [ ]:
def satisfy(X, free, passes):
    """Pull every thread toward its rest length, `passes` times. Changes X."""
    for _ in range(passes):
        d = X[E[:, 1]] - X[E[:, 0]]
        length = np.linalg.norm(d, axis=1)
        correction = (0.5 * (length - rest) / np.maximum(length, 1e-12))[:, None] * d
        move = np.zeros_like(X)
        asked = np.zeros(len(X))
        np.add.at(move, E[:, 0], correction)
        np.add.at(move, E[:, 1], -correction)
        np.add.at(asked, E.ravel(), 1)
        X[free] += (1.8 * move / np.maximum(asked, 1)[:, None])[free]       # 1.8: a little over-eager converges faster
    return X


def stretch(X):
    """The worst thread: how far from its rest length, as a fraction."""
    length = np.linalg.norm(X[E[:, 1]] - X[E[:, 0]], axis=1)
    return float(np.max(np.abs(length / rest - 1)))


# pull one corner away and watch the passes repair it
X = X0.copy()
X[0] += [-1.0, -1.0, 0.0]
everyone = np.ones(len(X), dtype=bool)
errors = [stretch(X)]
for _ in range(30):
    satisfy(X, everyone, 1)
    errors.append(stretch(X))
print(f"worst thread: {errors[0]:.0%} too long at first, {errors[10]:.1%} after 10 passes, {errors[30]:.2%} after 30")
chart({"worst stretch": errors}, "one corner pulled out: passes pull it back", xlabel="passes")

## 4 · Gravity, a sphere, a floor

Now put the pieces in one loop. Each time step:

1. gravity changes the velocities, velocities move the particles (Step 1)
2. the threads are satisfied, several passes (Step 3)
3. anything inside the sphere is pushed back out to its surface;
   anything below the floor is lifted onto it
4. the new velocity is simply *how far each particle actually moved*

Step 4 is the trick that makes it hold together: collisions and threads
change positions, and the velocity is read off afterwards, so it always
agrees with what happened.

In [ ]:
def simulate(X, steps=240, passes=25, pinned=(), sphere=1.02, floor=-0.6, keep=()):
    X = X.copy()
    V = np.zeros_like(X)
    free = np.ones(len(X), dtype=bool)
    free[list(pinned)] = False
    frames = {}
    for step in range(steps):
        before = X.copy()
        V[:, 2] += gravity * dt
        X[free] += 0.995 * V[free] * dt
        satisfy(X, free, passes)
        if sphere:
            distance = np.linalg.norm(X, axis=1)
            inside = (distance < sphere) & free
            X[inside] *= (sphere / distance[inside])[:, None]
        X[:, 2] = np.where(free, np.maximum(X[:, 2], floor), X[:, 2])
        V = (X - before) / dt
        if step in keep:
            frames[step] = X.copy()
    return (X, frames) if keep else X


def as_mesh(X):
    mesh = start.copy()
    for k, xyz in zip(keys, X):
        mesh.vertex_attributes(k, "xyz", [float(c) for c in xyz])
    return mesh


draped, frames = simulate(X0, keep=(10, 40, 239))
row(*[draw_mesh(as_mesh(frames[s]), f"t = {s / 60:.2f} s", view="iso", vertex_labels=False, size=(320, 300))
      for s in (10, 40, 239)])

In [ ]:
from compas.geometry import Sphere

final = as_mesh(draped)
view3d(Sphere(radius=0.98), (final, {"show_edges": True}))

Same 289 vertices, same faces, same keys in every frame. Only positions
changed: notebook 03 §2, sixty times a second.

In [ ]:
print("euler before and after:", start.euler(), final.euler())
print("vertex 0's neighbours before and after:", start.vertex_neighbors(0), final.vertex_neighbors(0))

## 5 · The honesty check

It looks like cloth. Is it? Real fabric barely stretches. The threads were
*asked* to keep their length; whether they did depends on how many passes
they were given. Measure it.

In [ ]:
print("passes   worst stretch")
results = {}
for passes in (2, 5, 10, 25, 50):
    results[passes] = stretch(simulate(X0, passes=passes))
    print(f"  {passes:3d}      {results[passes]:6.1%}")

With 2 passes the "cloth" is rubber: a thread somewhere is far longer
than it should be, and the picture would still have looked plausible.
**A simulation that was never measured is a picture.** The number of
passes is the price of accuracy, and now it has a figure attached.

In [ ]:
loose, tight = simulate(X0, passes=2), simulate(X0, passes=50)
row(draw_mesh(as_mesh(loose), f"2 passes: stretch {results[2]:.0%}", view="iso", vertex_labels=False, size=(440, 330)),
    draw_mesh(as_mesh(tight), f"50 passes: stretch {results[50]:.1%}", view="iso", vertex_labels=False, size=(440, 330)))

## 6 · Pin the corners: a hanging canopy

Take the sphere away and hold the four corners still. The cloth falls
until its threads stop it. What is left hanging is a **form found by
simulation**: all tension, like the hanging net of notebook 07.

In [ ]:
corners = [index[k] for k in start.vertices() if start.vertex_degree(k) == 2]
print("passes   sag of the middle   worst stretch")
for passes in (25, 60, 120):
    X = simulate(X0, steps=400, passes=passes, pinned=corners, sphere=None, floor=-99)
    print(f"  {passes:3d}        {1.6 - X[:, 2].min():.2f} m            {stretch(X):5.1%}")
hung = as_mesh(X)                                   # the last, most careful run
lo = min(hung.vertices_attribute("z"))
colours = {f: ramp((hung.face_centroid(f)[2] - lo) / (1.6 - lo)) for f in hung.faces()}
draw_mesh(hung, "held at four corners", view="iso", vertex_labels=False, face_colors=colours, size=(560, 400))

In [ ]:
view3d((hung, {"show_edges": True}))

Read that table before believing the picture. With 25 passes the middle
hangs far lower than with 120, and the difference is all **stretch**: four
corner threads now carry the whole cloth, and they need many more passes
to hold their length than the cloth on the sphere did. The sag is only an
answer once it stops changing when you ask for more passes. Section 5's
check was not a formality: here it changes the result by a third.

Compare the two ways to the same kind of shape:

| | Force density (notebook 07) | Falling cloth (this one) |
| --- | --- | --- |
| how | one linear solve | hundreds of small steps |
| you control | force per metre in each edge | the **length** of each edge |
| so the result | has the forces you asked for, lengths come out as they may | has the lengths you cut (as closely as the passes allow), forces come out as they may |
| use it when | designing a net or a shell from scratch | the material is given: a fabric of known size, a chain net, a membrane cut from flat sheets |

Turn this canopy upside down and you have a shell that stands in
compression, found from a square of cloth you could actually cut.

## What this used

One array of positions, one list of edges taken from the mesh, and the
rule "move, then repair". The mesh did the bookkeeping at the two ends:
it supplied the threads, and it took the result back with every key,
face and attribute still in place.

## Your turn
1. Pin only **two** opposite corners. Before running: what shape?
2. Make the cloth finer (`n=32`) and keep 25 passes. What happens to the
   stretch? Why does a finer mesh need more passes?
3. Replace the sphere by a box: push particles out through the nearest
   face. Which line of `simulate` changes?
4. Flip the hung canopy (`z → 3.2 − z`) and analyse it with `analyse`
   from [`08_mesh_as_truss`](08_mesh_as_truss.ipynb). Is it all compression?